In [ ]:
import os
from dotenv import load_dotenv

from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.ai.ml import load_component
from azure.ai.ml import dsl, Input, Output

load_dotenv()
src_dir = "../src"

In [2]:
# Get a handle to the workspace
credential = DefaultAzureCredential()

ml_client = MLClient(
    credential=credential,
    subscription_id=os.environ["AZURE_SUBSCRIPTION_ID"],
    resource_group_name=os.environ["AZURE_RESOURCE_GROUP"],
    workspace_name=os.environ["AZURE_WORKSPACE_NAME"],
)

GenAI tracing is not enabled. Set environment variable AZURE_EXPERIMENTAL_ENABLE_GENAI_TRACING=true to enable this experimental feature.
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [3]:
# Loading the component from the yml file
data_prep_component = load_component(source=os.path.join(src_dir, "components/data_prep/data_prep.yaml"))
train_component = load_component(source=os.path.join(src_dir, "components/train/train.yaml"))

In [4]:
# Now register the component to the workspace
data_prep_component = ml_client.create_or_update(data_prep_component)
train_component = ml_client.create_or_update(train_component)

Uploading data_prep (0.0 MBs): 100%|██████████| 1973/1973 [00:00<00:00, 3737.95it/s] 


Uploading train (0.0 MBs): 100%|██████████| 3261/3261 [00:00<00:00, 5808.59it/s] 




In [6]:
@dsl.pipeline(
    compute="serverless",  # "serverless" value runs pipeline on serverless compute
    description="E2E data_perp-train pipeline",
)
def credit_defaults_pipeline(
    pipeline_job_data_input,
    pipeline_job_test_train_ratio,
    pipeline_job_learning_rate,
    pipeline_job_registered_model_name,
):
    # using data_prep_function like a python call with its own inputs
    data_prep_job = data_prep_component(
        data=pipeline_job_data_input,
        test_train_ratio=pipeline_job_test_train_ratio,
    )

    # using train_func like a python call with its own inputs
    train_job = train_component(
        train_data=data_prep_job.outputs.train_data,  # note: using outputs from previous step
        test_data=data_prep_job.outputs.test_data,  # note: using outputs from previous step
        learning_rate=pipeline_job_learning_rate,  # note: using a pipeline input as parameter
        registered_model_name=pipeline_job_registered_model_name,
    )

    # a pipeline returns a dictionary of outputs
    # keys will code for the pipeline output identifier
    return {
        "pipeline_job_train_data": data_prep_job.outputs.train_data,
        "pipeline_job_test_data": data_prep_job.outputs.test_data,
    }

In [7]:
data_asset = ml_client.data.get(name="credit-card", version="v1")

In [8]:
registered_model_name = "credit_defaults_model"

# Let's instantiate the pipeline with the parameters of our choice
pipeline = credit_defaults_pipeline(
    pipeline_job_data_input=Input(type="uri_file", path=data_asset.path),
    pipeline_job_test_train_ratio=0.25,
    pipeline_job_learning_rate=0.05,
    pipeline_job_registered_model_name=registered_model_name,
)

In [9]:
# submit the pipeline job
pipeline_job = ml_client.jobs.create_or_update(
    pipeline,
    experiment_name="e2e_registered_components",
)
ml_client.jobs.stream(pipeline_job.name)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
pathOnCompute is not a known attribute

RunId: olden_lion_bbds5kkzxf
Web View: https://ml.azure.com/runs/olden_lion_bbds5kkzxf?wsid=/subscriptions/26d53a76-12ff-402e-977c-94c23577a60b/resourcegroups/demo/workspaces/demo-pipeline

Streaming logs/azureml/executionlogs.txt

[2026-05-08 06:52:13Z] Submitting 1 runs, first five are: 152db942:58671953-3c74-4f0f-85e7-6829b622a3ea
[2026-05-08 06:58:46Z] Execution of experiment failed, update experiment status and cancel running nodes.

Execution Summary
RunId: olden_lion_bbds5kkzxf
Web View: https://ml.azure.com/runs/olden_lion_bbds5kkzxf?wsid=/subscriptions/26d53a76-12ff-402e-977c-94c23577a60b/resourcegroups/demo/workspaces/demo-pipeline


JobException: Exception : 
 {
    "error": {
        "code": "UserError",
        "message": "Pipeline has failed child jobs. Failed nodes: /data_prep_job. For more details and logs, please go to the job detail page and check the child jobs.",
        "message_format": "Pipeline has failed child jobs. {0}",
        "message_parameters": {},
        "reference_code": "PipelineHasStepJobFailed",
        "details": []
    },
    "environment": "centralindia",
    "location": "centralindia",
    "time": "2026-05-08T06:58:46.244195Z",
    "component_name": ""
} 